# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- Conduct EDA: visualization and statistical measures to systematically understand the structure of the data.
- Recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- Inspect NaNs, datatypes, and summary statistics

In [2]:
#Load the data as aviation_df:
aviation_df = pd.read_csv("data/AviationData.csv", encoding = "cp1252")
aviation_df.head()

/var/folders/1j/rlzz05dn0b3b9mk9qtjndhy00000gn/T/ipykernel_30179/4162448126.py:2: DtypeWarning: Columns (6,7,28) have mixed types. Specify dtype option on import or set low_memory=False.
  aviation_df = pd.read_csv("data/AviationData.csv", encoding = "cp1252")


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


I can already see there are many differnet data types and NaNs... Let's see more info.

In [3]:
aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 88889 entries, 0 to 88888
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Event.Id                88889 non-null  object 
 1   Investigation.Type      88889 non-null  object 
 2   Accident.Number         88889 non-null  object 
 3   Event.Date              88889 non-null  object 
 4   Location                88837 non-null  object 
 5   Country                 88663 non-null  object 
 6   Latitude                34382 non-null  object 
 7   Longitude               34373 non-null  object 
 8   Airport.Code            50132 non-null  object 
 9   Airport.Name            52704 non-null  object 
 10  Injury.Severity         87889 non-null  object 
 11  Aircraft.damage         85695 non-null  object 
 12  Aircraft.Category       32287 non-null  object 
 13  Registration.Number     87507 non-null  object 
 14  Make                    88826 non-null

Oof, there's a lot of missing data in MANY columns... too many to even list here. Some columns look to have over 70,000 NaNs!

In [4]:
aviation_df.isna()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,False,False,False,False,False,False,True,True,True,True,...,False,True,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,True,True,True,True,...,False,True,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False,True,True,...,False,True,False,True,True,True,False,False,False,False
3,False,False,False,False,False,False,True,True,True,True,...,False,True,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,True,True,True,True,...,False,True,False,False,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
88884,False,False,False,False,False,False,True,True,True,True,...,False,True,False,False,False,False,True,True,True,False
88885,False,False,False,False,False,False,True,True,True,True,...,True,True,False,False,False,False,True,True,True,True
88886,False,False,False,False,False,False,False,False,False,False,...,False,True,False,False,False,False,False,True,True,False
88887,False,False,False,False,False,False,True,True,True,True,...,False,False,False,False,False,False,True,True,True,True


Lots of missing data in the `Latitude`, `Longitude`, `Airport.Code`, `Airport.Name`, `Air.carrier`, and other columns.

In [5]:
aviation_df.describe()

,Number.of.Engines,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured
count,82805.000000,77488.000000,76379.000000,76956.000000,82977.000000
mean,1.146585,0.647855,0.279881,0.357061,5.325440
std,0.446510,5.485960,1.544084,2.235625,27.913634
min,0.000000,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000,1.000000
75%,1.000000,0.000000,0.000000,0.000000,2.000000
max,8.000000,349.000000,161.000000,380.000000,699.000000


From just these statistics, it seems like the total number of fatal and serious injuries is 0 at least 75% of the time, but the max number is 349 fatalities, which is really bad. I can't tell yet if 349 is an outlier, but it looks good to notice that most of the time, there are 0 fatal injuries. Even serious and minor injuries are 0 at least 75% of the time.

## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- Inspect relevant columns
- Figure out any reasonable imputations
- Filter the dataset

In [6]:
#See what values occur in the "Aircraft.Category" column:
aviation_df["Aircraft.Category"].value_counts()

Aircraft.Category
Airplane             27617
Helicopter            3440
Glider                 508
Balloon                231
Gyrocraft              173
Weight-Shift           161
Powered Parachute       91
Ultralight              30
Unknown                 14
WSFT                     9
Powered-Lift             5
Blimp                    4
UNK                      2
Rocket                   1
ULTR                     1
Name: count, dtype: int64

In [7]:
#We only want rows with "Airplane" as the entry:
aviation_df = aviation_df[aviation_df["Aircraft.Category"] == "Airplane"]
#Check that it worked:
aviation_df["Aircraft.Category"]

5        Airplane
7        Airplane
8        Airplane
12       Airplane
13       Airplane
           ...   
88869    Airplane
88873    Airplane
88876    Airplane
88877    Airplane
88886    Airplane
Name: Aircraft.Category, Length: 27617, dtype: object

In [8]:
#See what values occur in the "Amateur.Built" column:
aviation_df["Amateur.Built"].value_counts()

Amateur.Built
No     24417
Yes     3183
Name: count, dtype: int64

In [9]:
#We only want rows with "No" as the entry because we only want professional
#builds:
aviation_df = aviation_df[aviation_df["Amateur.Built"] == "No"]
#Check if it worked:
aviation_df["Amateur.Built"]

5        No
7        No
8        No
12       No
13       No
         ..
88869    No
88873    No
88876    No
88877    No
88886    No
Name: Amateur.Built, Length: 24417, dtype: object

In [10]:
#First, check what dtype the column "Event.Date" is:
aviation_df["Event.Date"].dtype

dtype('O')

In [11]:
#We don't want it to be an object, we want datetime, so let's convert it:
aviation_df["Event.Date"] = pd.to_datetime(aviation_df["Event.Date"])
#Let's check if that worked:
aviation_df["Event.Date"].dtype

dtype('<M8[ns]')

In [12]:
#Awesome, now we only want rows with dates from 1983 or later:
aviation_df = aviation_df[aviation_df["Event.Date"].dt.year >= 1983]
#Check if it worked:
aviation_df["Event.Date"]

4149    1983-03-18
4150    1983-03-18
4171    1983-03-20
4285    1983-04-02
5957    1983-08-21
           ...    
88869   2022-12-13
88873   2022-12-14
88876   2022-12-15
88877   2022-12-16
88886   2022-12-26
Name: Event.Date, Length: 21447, dtype: datetime64[ns]

### Cleaning and Constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct a metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious/fatal injury can be estimated as a fraction from this.

In [13]:
#First, I need to see what columns are in the dataset:
aviation_df.columns

Index(['Event.Id', 'Investigation.Type', 'Accident.Number', 'Event.Date',
       'Location', 'Country', 'Latitude', 'Longitude', 'Airport.Code',
       'Airport.Name', 'Injury.Severity', 'Aircraft.damage',
       'Aircraft.Category', 'Registration.Number', 'Make', 'Model',
       'Amateur.Built', 'Number.of.Engines', 'Engine.Type', 'FAR.Description',
       'Schedule', 'Purpose.of.flight', 'Air.carrier', 'Total.Fatal.Injuries',
       'Total.Serious.Injuries', 'Total.Minor.Injuries', 'Total.Uninjured',
       'Weather.Condition', 'Broad.phase.of.flight', 'Report.Status',
       'Publication.Date'],
      dtype='object')

In [14]:
#Maybe we can calculate how many total passengers were on each flight by adding
#together the "Total.Fatal.Injuries", "Total.Serious.Injuries",
#"Total.Minor.Injuries", and "Total.Uninjured" columns. Let's create a new
#column for this:
aviation_df["Total.Passengers"] = aviation_df[
    ["Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"]
    #Using .sum() ensures NaNs are skipped and axis = 1 ensures we are summing
    #across columns:
    ].sum(axis = 1)

#Check changes:
aviation_df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,NaN,NaN,NaN,NaN,588.0,VMC,Standing,Probable Cause,04-12-2014,588.0
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,"Singapore Airlines, Ltd.",NaN,NaN,NaN,588.0,VMC,Taxi,Probable Cause,04-12-2014,588.0
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,NaN,1.0,1.0,NaN,NaN,IMC,Cruise,Probable Cause,02-05-2011,2.0
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,NaN,1.0,NaN,NaN,4.0,VMC,Standing,Probable Cause,17-10-2016,5.0
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,289.0,VMC,Cruise,Probable Cause,01-02-2016,289.0


In [15]:
#For injured passengers of any kind, NaN probably means 0 injuries recorded, so
#let's replace NaNs in injury columns with 0s:
cols = [
    "Total.Passengers",
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured"
]

aviation_df[cols] = aviation_df[cols].fillna(0)

In [16]:
#Now let's calculate how many people on each flight had fatal or serious #injuries by calculating that fraction of total passengers on each flight:
aviation_df["Serious.or.Fatal.Fraction"] = (
    (aviation_df["Total.Fatal.Injuries"] +
     aviation_df["Total.Serious.Injuries"]) /
    aviation_df["Total.Passengers"]
)

#Check changes:
aviation_df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers,Serious.or.Fatal.Fraction
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,0.0,0.0,0.0,588.0,VMC,Standing,Probable Cause,04-12-2014,588.0,0.0
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,0.0,0.0,0.0,588.0,VMC,Taxi,Probable Cause,04-12-2014,588.0,0.0
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,1.0,1.0,0.0,0.0,IMC,Cruise,Probable Cause,02-05-2011,2.0,1.0
4285,20001214X42672,Accident,FTW83LA177,1983-04-02,"MCKINNEY, TX",United States,NaN,NaN,TX05,AERO COUNTRY,...,1.0,0.0,0.0,4.0,VMC,Standing,Probable Cause,17-10-2016,5.0,0.2
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,0.0,289.0,VMC,Cruise,Probable Cause,01-02-2016,289.0,0.0


In [17]:
#Let's check if the "Serious.or.Fatal.Fraction" column has any NaNs (infinity
#after calculating that fraction):
print(aviation_df["Serious.or.Fatal.Fraction"].isna().sum())

904


In [18]:
#That's a lot of NaNs. Let's see what rows are causing those NaNs:
aviation_df[aviation_df["Serious.or.Fatal.Fraction"].isna()][
["Total.Passengers", "Total.Fatal.Injuries", "Total.Serious.Injuries"]
].head()

,Total.Passengers,Total.Fatal.Injuries,Total.Serious.Injuries
29591,0.0,0.0,0.0
41656,0.0,0.0,0.0
42287,0.0,0.0,0.0
42613,0.0,0.0,0.0
42847,0.0,0.0,0.0


Looks like the NaNs are caused by rows where there are 0 injuries, but also "Total.Passengers" is 0, and 0/0 = infinity = NaN. For the purpose of figuring out how many flights had serious or fatal injuries, 0 injuries / 0 people is stil a 0% injury rate. So I'm going to fill those NaNs in the `Serious.or.Fatal.Fraction` column with 0s.

In [19]:
aviation_df["Serious.or.Fatal.Fraction"] = (
    aviation_df["Serious.or.Fatal.Fraction"].fillna(0))
#Let's check if it worked:
aviation_df["Serious.or.Fatal.Fraction"].isna().sum()

0

Awesome! Now we have a colum with the total fraction of passengers with serious or fatal injuries, and that column has no NaNs! We can see the first five lines of that column below:

In [20]:
aviation_df["Serious.or.Fatal.Fraction"].head()

4149    0.0
4150    0.0
4171    1.0
4285    0.2
5957    0.0
Name: Serious.or.Fatal.Fraction, dtype: float64

**Aircraft.Damage**
- Identify and execute any cleaning tasks.
- Construct a derived column tracking whether an aircraft was destroyed or not.

In [21]:
#First, let's look at some info about the "Aircraft.damage" column:
aviation_df["Aircraft.damage"].head(15)

4149           Minor
4150           Minor
4171       Destroyed
4285             NaN
5957           Minor
5960       Destroyed
6669             NaN
6760           Minor
6806     Substantial
7084     Substantial
7708     Substantial
8585     Substantial
8591     Substantial
8865       Destroyed
10140          Minor
Name: Aircraft.damage, dtype: object

In [22]:
#There are some NaNs here. Let's check the value_counts() to see if we have
#enough data for dropping:
aviation_df["Aircraft.damage"].value_counts()

Aircraft.damage
Substantial    16990
Destroyed       2316
Minor            817
Unknown           97
Name: count, dtype: int64

In [23]:
#We only REALLY care about aircraft that are destroyed, so I'm going to remove
#rows with NaNs in the "Aircraft.damage" column because we have enough
#representation from other categories with actual data:
aviation_df = aviation_df.dropna(subset=["Aircraft.damage"])
#Check if it worked:
aviation_df["Aircraft.damage"].isna().sum()

0

In [24]:
#Now, let's create a derived column that tracks whether an aircraft was
#destroyed or not:
aviation_df["Aircraft.Destroyed"] = (
    aviation_df["Aircraft.damage"] == "Destroyed"
)
#Let's check our dataset so far:
aviation_df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date,Total.Passengers,Serious.or.Fatal.Fraction,Aircraft.Destroyed
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,0.0,0.0,588.0,VMC,Standing,Probable Cause,04-12-2014,588.0,0.0,False
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,NaN,NaN,LAX,LOS ANGELES INTL,...,0.0,0.0,588.0,VMC,Taxi,Probable Cause,04-12-2014,588.0,0.0,False
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,NaN,NaN,NaN,NaN,...,1.0,0.0,0.0,IMC,Cruise,Probable Cause,02-05-2011,2.0,1.0,True
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,NaN,NaN,NaN,NaN,...,0.0,0.0,289.0,VMC,Cruise,Probable Cause,01-02-2016,289.0,0.0,False
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,NaN,NaN,S88,NaN,...,2.0,0.0,13.0,VMC,Other,Probable Cause,17-10-2016,26.0,0.5,True


### Investigate the *Make* column
- Identify cleaning tasks here.
- List cleaning tasks clearly in markdown.
- Execute the cleaning tasks.
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50, though lower could work as well).

In [25]:
aviation_df["Make"].info()

<class 'pandas.core.series.Series'>
Index: 20220 entries, 4149 to 88886
Series name: Make
Non-Null Count  Dtype 
--------------  ----- 
20219 non-null  object
dtypes: object(1)
memory usage: 315.9+ KB


Ok, not a lot of NaNs. That's good. Let's look at part of the column, and then check the value_counts:

In [26]:
aviation_df["Make"].head(15)

4149       Lockheed
4150         Boeing
4171          Piper
5957        Douglas
5960       Lockheed
6760         Boeing
6806          Beech
7084         Cessna
7708          Beech
8585          Piper
8591       Lockheed
8865          Piper
10140    Swearingen
10247        Cessna
10605        Cessna
Name: Make, dtype: object

In [27]:
aviation_df["Make"].value_counts().head(50)

Make
CESSNA                            4786
PIPER                             2775
Cessna                            2259
Piper                             1180
BEECH                             1002
BOEING                             534
Beech                              408
MOONEY                             235
AIR TRACTOR INC                    217
CIRRUS DESIGN CORP                 215
BELLANCA                           157
AERONCA                            149
Boeing                             146
MAULE                              144
Mooney                             124
Air Tractor                        117
LUSCOMBE                            93
STINSON                             91
CHAMPION                            90
AIR TRACTOR                         89
DEHAVILLAND                         88
AIRBUS                              82
NORTH AMERICAN                      77
EMBRAER                             76
CIRRUS                              76
GRUMMAN             

Oof ok, there are a lot of data entry issues causing the same Make to be counted multiple times (e.g. "CESSNA" and "Cessna" should be the same.) Less fix that.

In [28]:
#First, let's convert every entry in the column to uppercase and strip
#whitespace, which should help combine some of these columns:
aviation_df["Make"] = aviation_df["Make"].str.upper().str.strip()
#Now let's check the value_counts again:
aviation_df["Make"].value_counts().head(50)

Make
CESSNA                            7045
PIPER                             3955
BEECH                             1410
BOEING                             680
MOONEY                             359
AIR TRACTOR INC                    219
BELLANCA                           218
CIRRUS DESIGN CORP                 217
MAULE                              215
AIR TRACTOR                        206
AERONCA                            200
CHAMPION                           157
GRUMMAN                            145
LUSCOMBE                           139
CIRRUS                             132
STINSON                            129
NORTH AMERICAN                     104
EMBRAER                             94
TAYLORCRAFT                         93
DEHAVILLAND                         92
AIRBUS                              91
AERO COMMANDER                      89
AVIAT AIRCRAFT INC                  75
DIAMOND AIRCRAFT IND INC            74
SOCATA                              73
MCDONNELL DOUGLAS   

Much better, but still some companies that should be combined (e.g. "AIR TRACTOR INC" and "AIR TRACTOR"). Let's fix this:

In [29]:
aviation_df.loc[aviation_df["Make"].str.contains("CESSNA", na=False), "Make"] = "CESSNA"
aviation_df.loc[aviation_df["Make"].str.contains("PIPER", na=False), "Make"] = "PIPER"
aviation_df.loc[aviation_df["Make"].str.contains("BEECH", na=False), "Make"] = "BEECH"
aviation_df.loc[aviation_df["Make"].str.contains("BOEING", na=False), "Make"] = "BOEING"
aviation_df.loc[aviation_df["Make"].str.contains("MOONEY", na=False), "Make"] = "MOONEY"
aviation_df.loc[aviation_df["Make"].str.contains("AIR TRACTOR", na=False), "Make"] = "AIR TRACTOR"
aviation_df.loc[aviation_df["Make"].str.contains("GRUMMAN", na=False), "Make"] = "GRUMMAN"
aviation_df.loc[aviation_df["Make"].str.contains("AIRBUS", na=False), "Make"] = "AIRBUS"
aviation_df.loc[aviation_df["Make"].str.contains("ROCKWELL", na=False), "Make"] = "ROCKWELL"
aviation_df.loc[aviation_df["Make"].str.contains("AVIAT", na=False), "Make"] = "AVIAT"
aviation_df.loc[aviation_df["Make"].str.contains("CIRRUS", na=False), "Make"] = "CIRRUS"
aviation_df.loc[aviation_df["Make"].str.contains("AYRES", na=False), "Make"] = "AYRES"
aviation_df.loc[aviation_df["Make"].str.contains("DE HAVILLAND", na=False), "Make"] = "DEHAVILLAND"
aviation_df.loc[aviation_df["Make"].str.contains("FLIGHT DESIGN", na=False), "Make"] = "FLIGHT DESIGN"
#Now let's check the value_counts again:
aviation_df["Make"].value_counts().head(50)

Make
CESSNA                            7101
PIPER                             4022
BEECH                             1480
BOEING                             694
AIR TRACTOR                        432
MOONEY                             407
CIRRUS                             390
GRUMMAN                            309
AVIAT                              253
BELLANCA                           218
MAULE                              215
AERONCA                            200
DEHAVILLAND                        157
CHAMPION                           157
LUSCOMBE                           139
STINSON                            129
AIRBUS                             114
NORTH AMERICAN                     104
ROCKWELL                           103
AYRES                               99
EMBRAER                             94
TAYLORCRAFT                         93
AERO COMMANDER                      89
FLIGHT DESIGN                       77
DIAMOND AIRCRAFT IND INC            74
SOCATA              

Ok, now let's keep the Makes with more than 50 counts to look at later, and group everything else into "OTHER":

In [30]:
make_counts = aviation_df["Make"].value_counts()
valid_makes = make_counts[make_counts > 50].index
aviation_df["Make"] = aviation_df["Make"].apply(
    lambda x: x if x in valid_makes else "OTHER"
)
#Check if it worked:
aviation_df["Make"].value_counts()

Make
CESSNA                        7101
PIPER                         4022
OTHER                         2759
BEECH                         1480
BOEING                         694
AIR TRACTOR                    432
MOONEY                         407
CIRRUS                         390
GRUMMAN                        309
AVIAT                          253
BELLANCA                       218
MAULE                          215
AERONCA                        200
DEHAVILLAND                    157
CHAMPION                       157
LUSCOMBE                       139
STINSON                        129
AIRBUS                         114
NORTH AMERICAN                 104
ROCKWELL                       103
AYRES                           99
EMBRAER                         94
TAYLORCRAFT                     93
AERO COMMANDER                  89
FLIGHT DESIGN                   77
DIAMOND AIRCRAFT IND INC        74
SOCATA                          73
MCDONNELL DOUGLAS               73
RAYTHEON AIRCRA

### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [31]:
#First, get rid of any NaNs in the "Model" column:
aviation_df = aviation_df.dropna(subset=["Model"])

Now, check if model names repeat across different airplane makers.
In other words, if I just say "737", does that automatically tell me that it's a Boeing, or do I need to say "Boeing 737"? Likely it's the latter, but let's check.

#### Here's how to read the output of the below code:

Model   → Number of Different Makes Who Use that Model Name

172     → 1

737     → 1

PA28    → 1

MODEL_X → 3 different makes

In [32]:
aviation_df.groupby("Model")["Make"].nunique().sort_values(ascending = False).head(20)

Model
500      5
7ECA     4
S2R      4
7GCAA    4
8GCBC    4
8KCAB    4
7GCBC    4
7AC      4
G36      4
300      3
B200     3
7KCAB    3
AA5      3
400A     3
400      3
200      3
58       3
390      3
F19      3
7EC      3
Name: Make, dtype: int64

Ok so clearly most model names are used by many different airplane makers, which makes sense. This means we need to create a derived column that is a unique identifier for each plane type (i.e. Boeing 737 is different from a Cessna 737).

In [33]:
aviation_df["Make.Model"] = aviation_df["Make"] + " " + aviation_df["Model"]
aviation_df["Make.Model"].head()

4149               OTHER L-1011
4150                 BOEING 747
4171            PIPER PA-28-140
5957             OTHER DC-10-10
5960    OTHER LEARSTAR, L-18-56
Name: Make.Model, dtype: object

### Cleaning Other Columns
There are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks.

**Hint**: *Some things you might want to think about are values in numerical features that do not make sense or categories that contain too few examples. Consider treating values (for either categorical or numeric data) that might be placeholders for NaNs.*

**Note**: You do not necessarily need to impute or drop NaNs here.

#### Inspecting and Cleaning `Engine.Type`

In [34]:
#Insepct the value_counts of Engine.Type:
aviation_df["Engine.Type"].value_counts()

Engine.Type
Reciprocating      14934
Turbo Prop          1197
Turbo Fan            570
Unknown              104
Turbo Jet            103
Turbo Shaft           10
Electric               5
Geared Turbofan        1
UNK                    1
Name: count, dtype: int64

In [35]:
#Data cleaning of Engine.Type:
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].str.strip()
aviation_df["Engine.Type"] = aviation_df["Engine.Type"].replace("UNK", "Unknown")
#Check if it worked:
aviation_df["Engine.Type"].value_counts()

Engine.Type
Reciprocating      14934
Turbo Prop          1197
Turbo Fan            570
Unknown              105
Turbo Jet            103
Turbo Shaft           10
Electric               5
Geared Turbofan        1
Name: count, dtype: int64

#### Inspecting and Cleaning `Weather.Condition`

In [36]:
#Inspect the value_counts of Weather.Condition:
aviation_df["Weather.Condition"].value_counts()

Weather.Condition
VMC    16756
IMC      999
Unk      167
UNK       69
Name: count, dtype: int64

In [37]:
#Data cleaning of Weather.Condition:
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].str.strip()
aviation_df["Weather.Condition"] = aviation_df["Weather.Condition"].replace("Unk", "UNK")
#Check if it worked:
aviation_df["Weather.Condition"].value_counts()

Weather.Condition
VMC    16756
IMC      999
UNK      236
Name: count, dtype: int64

#### Inspecting and Cleaning `Number.of.Engines`

In [38]:
#Inspect the value_counts of Number.of.Engines:
aviation_df["Number.of.Engines"].value_counts()

Number.of.Engines
1.0    15646
2.0     2464
4.0       66
3.0       33
0.0        7
8.0        1
6.0        1
Name: count, dtype: int64

In [39]:
#Data cleaning of Number.of.Engines:
#We can only have a Number.of.Engines greater than 0, so let's turn all zeros
#into NaNs so they don't mess up our data later:
aviation_df.loc[aviation_df["Number.of.Engines"] == 0, "Number.of.Engines"] = pd.NA
#Check if it worked:
aviation_df["Number.of.Engines"].value_counts()

Number.of.Engines
1.0    15646
2.0     2464
4.0       66
3.0       33
8.0        1
6.0        1
Name: count, dtype: int64

#### Inspecting and Cleaning `Purpose.of.flight`

In [40]:
#Inspect the value_counts of Purpose.of.flight:
aviation_df["Purpose.of.flight"].value_counts()

Purpose.of.flight
Personal                     11666
Instructional                 2731
Aerial Application             932
Business                       489
Positioning                    332
Unknown                        331
Skydiving                      160
Aerial Observation             160
Other Work Use                 152
Flight Test                    117
Ferry                          102
Executive/corporate             99
Banner Tow                      89
Public Aircraft - Federal       51
Air Race show                   45
Public Aircraft                 37
Glider Tow                      35
Public Aircraft - State         24
Firefighting                    16
Public Aircraft - Local         12
ASHO                             5
Air Race/show                    4
Air Drop                         3
PUBS                             3
Name: count, dtype: int64

In [41]:
#Data cleaning of Purpose.of.flight:
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].str.title().str.strip()
#Clean duplicates:
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].replace({
    "Air Race/show": "Air Race Show",
    "Air Race show": "Air Race Show",
    "Executive/corporate": "Executive/Corporate",
})
#Only keep flight purposes that occur over 100 times:
purpose_counts = aviation_df["Purpose.of.flight"].value_counts()
keep_purposes = purpose_counts[purpose_counts > 100].index
aviation_df["Purpose.of.flight"] = aviation_df["Purpose.of.flight"].apply(
    lambda x: x if x in keep_purposes else "Other"
)
#Check if it worked:
aviation_df["Purpose.of.flight"].value_counts()

Purpose.of.flight
Personal              11666
Other                  3032
Instructional          2731
Aerial Application      932
Business                489
Positioning             332
Unknown                 331
Skydiving               160
Aerial Observation      160
Other Work Use          152
Flight Test             117
Ferry                   102
Name: count, dtype: int64

#### Inspecting and Cleaning `Broad.phase.of.flight`

#Inspect the value_counts of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"].value_counts()


In [42]:
#Inspect the value_counts of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"].value_counts()

Broad.phase.of.flight
Landing        1255
Takeoff         506
Cruise          255
Approach        232
Maneuvering     166
Taxi            105
Go-around        89
Descent          65
Climb            53
Standing         37
Unknown          12
Other             3
Name: count, dtype: int64

In [43]:
#Data cleaning of Broad.phase.of.flight:
aviation_df["Broad.phase.of.flight"] = aviation_df["Broad.phase.of.flight"].str.strip()
#Only keep flight phases that appear more than 50 times:
phase_counts = aviation_df["Broad.phase.of.flight"].value_counts()
keep_phases = phase_counts[phase_counts > 50].index
aviation_df["Broad.phase.of.flight"] = aviation_df["Broad.phase.of.flight"].apply(lambda x: x if x in keep_phases else "Other")
#Check if it worked:
aviation_df["Broad.phase.of.flight"].value_counts()

Broad.phase.of.flight
Other          17478
Landing         1255
Takeoff          506
Cruise           255
Approach         232
Maneuvering      166
Taxi             105
Go-around         89
Descent           65
Climb             53
Name: count, dtype: int64

### Column Removal
Inspect the DataFrame and drop any columns that have too many NaNs. For this exercise, keep all columns with more than 20,000 non-nulls.

In [44]:
#First, inspect the DataFrame to figure out how many non-null values are in
#each column:
aviation_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20204 entries, 4149 to 88886
Data columns (total 35 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Event.Id                   20204 non-null  object        
 1   Investigation.Type         20204 non-null  object        
 2   Accident.Number            20204 non-null  object        
 3   Event.Date                 20204 non-null  datetime64[ns]
 4   Location                   20200 non-null  object        
 5   Country                    20203 non-null  object        
 6   Latitude                   18611 non-null  object        
 7   Longitude                  18604 non-null  object        
 8   Airport.Code               13580 non-null  object        
 9   Airport.Name               13654 non-null  object        
 10  Injury.Severity            19809 non-null  object        
 11  Aircraft.damage            20204 non-null  object        
 12  Aircra

In [45]:
#Now, drop any columns with less than 20,000 non-null values:
aviation_df = aviation_df.drop(columns = ["Latitude", "Longitude", "Airport.Code", "Airport.Name", "Injury.Severity", "FAR.Description", "Schedule", "Air.carrier", "Report.Status", "Publication.Date"])
#I kept Weather.Condition because the assignment told us it will be useful later.
#Now, for the last time, let's take a look at the dataset's head with what remains:
aviation_df.head()

,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Aircraft.damage,Aircraft.Category,Registration.Number,Make,...,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Total.Passengers,Serious.or.Fatal.Fraction,Aircraft.Destroyed,Make.Model
4149,20001214X42478,Incident,LAX83IA149B,1983-03-18,"LOS ANGELES, CA",United States,Minor,Airplane,N323EA,OTHER,...,0.0,0.0,0.0,588.0,VMC,Other,588.0,0.0,False,OTHER L-1011
4150,20001214X42478,Incident,LAX83IA149A,1983-03-18,"LOS ANGELES, CA",United States,Minor,Airplane,9VSQQ,BOEING,...,0.0,0.0,0.0,588.0,VMC,Taxi,588.0,0.0,False,BOEING 747
4171,20001214X42331,Accident,ATL83FA140,1983-03-20,"CROSSVILLE, TN",United States,Destroyed,Airplane,N9600W,PIPER,...,1.0,1.0,0.0,0.0,IMC,Cruise,2.0,1.0,True,PIPER PA-28-140
5957,20001214X44248,Incident,MIA83IA210,1983-08-21,"NORFOLK, VA",United States,Minor,Airplane,N69NA,OTHER,...,0.0,0.0,0.0,289.0,VMC,Cruise,289.0,0.0,False,OTHER DC-10-10
5960,20001214X44100,Accident,DCA83AA036,1983-08-21,"SILVANA, WA",United States,Destroyed,Airplane,N116CA,OTHER,...,11.0,2.0,0.0,13.0,VMC,Other,26.0,0.5,True,"OTHER LEARSTAR, L-18-56"


### Save DataFrame to CSV
- It's generally useful to save data to file/server after it's in a sufficiently cleaned or intermediate state.
- The data can then be loaded directly in another notebook for further analysis.
- This helps keep your notebooks and workflow readable, clean and modularized.

In [46]:
aviation_df.to_csv("data/AviationDataCleaned.csv", index = False)